In [33]:
import boto3
import os
import re
import pandas as pd
from itertools import product
from pathlib import Path

judged_scenarios_path: Path = Path.cwd() / 'data/judged_scenarios/'
data = []
for file in os.listdir(judged_scenarios_path):

    if file.endswith('.json'):
        match = re.search(r'batch-scenarios-(\d+)-policies-each-(.+)-temp(\d+\.\d+)\.json', file)
        if match:
            complexity = int(match.group(1))
            model = match.group(2).replace('_', ' ')
            temp = float(match.group(3))
            
            if model.startswith('200-'):
                model = model[4:]
                
            with open(judged_scenarios_path / file, 'r') as f:
                count = f.read().count('"scenario-id"')
            
            data.append([model, temp, complexity, count])

df = pd.DataFrame(data, columns=['Model', 'Temperature', 'Scenario Complexity', 'Num Scenarios'])
df = df.sort_values(['Model', 'Temperature', 'Scenario Complexity'])
print(df.to_markdown(index=False))

# Find missing combinations for Temperature 0
temp0_df = df[df['Temperature'] == 0.0]
models = temp0_df['Model'].unique()
complexities = [4, 6, 8, 10]

existing = set(zip(temp0_df['Model'], temp0_df['Scenario Complexity']))
all_combinations = set(product(models, complexities))
missing = all_combinations - existing

print("\n**Missing combinations for Temperature 0.0:**")
for model, complexity in sorted(missing):
    print(f"- {model}, Scenario Complexity {complexity}")


| Model             |   Temperature |   Scenario Complexity |   Num Scenarios |
|:------------------|--------------:|----------------------:|----------------:|
| claude 3 7 sonnet |           0   |                     4 |             200 |
| claude 3 7 sonnet |           0   |                     6 |             200 |
| claude 3 7 sonnet |           0   |                     8 |             200 |
| claude 3 7 sonnet |           0   |                    10 |             196 |
| claude 3 7 sonnet |           0.1 |                     4 |             200 |
| claude 3 7 sonnet |           0.1 |                     6 |             200 |
| claude 3 7 sonnet |           0.1 |                     8 |             200 |
| claude 3 7 sonnet |           0.1 |                    10 |             196 |
| claude 4 sonnet   |           0   |                     4 |             200 |
| claude 4 sonnet   |           0   |                     6 |             200 |
| claude 4 sonnet   |           0   |   

In [ ]:
#list inference profiles available in a given region

bedrock_client = boto3.client('bedrock', region_name='us-east-1')
response = bedrock_client.list_inference_profiles()
for profile in response['inferenceProfileSummaries']:
    if 'nova' in profile['inferenceProfileName'].lower():
        print(f"ARN: {profile['inferenceProfileArn']}")

In [27]:
import boto3
import fnmatch
from pathlib import Path
from collections import defaultdict

def list_s3_versions(bucket_name, wildcard_pattern):
    """List object versions matching wildcard pattern"""
    s3 = boto3.client('s3')
    versions = []
    
    paginator = s3.get_paginator('list_object_versions')
    for page in paginator.paginate(Bucket=bucket_name):
        for version in page.get('Versions', []):
            key = version['Key']
            if fnmatch.fnmatch(key, wildcard_pattern):
                versions.append({
                    'key': key,
                    'version_id': version['VersionId'],
                    'last_modified': version['LastModified'],
                    'size': version['Size']
                })
    
    return sorted(versions, key=lambda x: (x['key'], x['last_modified']))

def select_versions(versions):
    """Interactive selection of versions with grouped display"""
    # Group versions by key
    grouped = defaultdict(list)
    for v in versions:
        grouped[v['key']].append(v)
    
    print("\nAvailable versions:")
    index = 0
    for key, key_versions in grouped.items():
        # Green header with count
        print(f"\033[92m{key} ({len(key_versions)} versions)\033[0m")
        
        for v in key_versions:
            print(f"  {index:3d}: {v['version_id'][:8]}... | {v['last_modified']} | {v['size']} bytes")
            index += 1
        print()
    
    selection = input("Enter indices (comma-separated, e.g., 0,2,5): ")
    indices = [int(x.strip()) for x in selection.split(',') if x.strip().isdigit()]
    return [versions[i] for i in indices if 0 <= i < len(versions)]

def download_versions(bucket_name, selected_versions, local_folder):
    """Download selected versions to local folder"""
    s3 = boto3.client('s3')
    local_path = Path(local_folder).expanduser()
    local_path.mkdir(parents=True, exist_ok=True)
    
    for version in selected_versions:
        filename = f"{Path(version['key']).stem}_{version['version_id'][:8]}{Path(version['key']).suffix}"
        local_file = local_path / filename
        
        s3.download_file(
            bucket_name, 
            version['key'], 
            str(local_file),
            ExtraArgs={'VersionId': version['version_id']}
        )
        print(f"Downloaded: {local_file}")

# Usage
bucket_name = input("Enter S3 bucket name: ")
wildcard_pattern = input("Enter wildcard pattern (e.g., data/*.json): ")
local_folder = input("Enter local download folder: ")

versions = list_s3_versions(bucket_name, wildcard_pattern)
if versions:
    selected = select_versions(versions)
    if selected:
        download_versions(bucket_name, selected, local_folder)
        print(f"\nDownloaded {len(selected)} files to {local_folder}")
else:
    print("No matching versions found")



Enter S3 bucket name:  183023889407-us-east-1-compliance-rule-generator
Enter wildcard pattern (e.g., data/*.json):  scenarios-judged/judged_scenarios_batch-scenarios-*nova_2_lite*.json
Enter local download folder:  ~/compliance/data/judged_scenarios/temp/



Available versions:
scenarios-judged/judged_scenarios_batch-scenarios-10-policies-each-200-nova_2_lite-temp0.0.json (1 versions)
    0: 6_eFmV.V... | 2026-02-02 03:44:53+00:00 | 1014561 bytes

scenarios-judged/judged_scenarios_batch-scenarios-10-policies-each-200-nova_2_lite-temp0.1.json (1 versions)
    1: W3sMI5Xf... | 2026-03-03 04:12:41+00:00 | 1013623 bytes

scenarios-judged/judged_scenarios_batch-scenarios-10-policies-each-200-nova_2_lite-temp0.2.json (1 versions)
    2: VBZwjGbx... | 2026-01-27 05:54:31+00:00 | 1012770 bytes

scenarios-judged/judged_scenarios_batch-scenarios-10-policies-each-nova_2_lite-temp0.0.json (1 versions)
    3: VS6e0rik... | 2026-03-07 23:43:39+00:00 | 857361 bytes

scenarios-judged/judged_scenarios_batch-scenarios-10-policies-each-nova_2_lite-temp0.1.json (1 versions)
    4: KXT_J5wP... | 2026-03-08 00:35:20+00:00 | 862169 bytes

scenarios-judged/judged_scenarios_batch-scenarios-4-policies-each-nova_2_lite-temp0.0.json (2 versions)
    5: YwsW6kHz... |

Enter indices (comma-separated, e.g., 0,2,5):  
